# CaImAn Parameter Run Comparison

Loads multiple CaImAn runs from the same session, matches components across runs using
spatial footprint IoU (Hungarian algorithm), then compares:
- spatial layout (matched vs unmatched)
- confidence in the matching
- evaluation scores (r_value, SNR, CNN) for matched cells
- ΔF/F traces and residual noise

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from scipy.optimize import linear_sum_assignment

try:
    cv2.setNumThreads(0)
except():
    pass

from IPython import get_ipython
try:
    if __IPYTHON__:
        get_ipython().run_line_magic('load_ext', 'autoreload')
        get_ipython().run_line_magic('autoreload', '2')
except NameError:
    pass

import caiman as cm
from caiman.source_extraction.cnmf import cnmf

%matplotlib inline

## 1. Load CaImAn Runs

In [ ]:
BASE_DIR           = "/media/toor/Seagate3/YOUNG_MICE/Mouse946/test_cropping"
RUN_NAMES          = [ "caiman_final_1215_rf48", "caiman_final_1215", "caiman_final_1015"]
IOU_THRESHOLD      = 0.3    # min IoU to link two components as the same cell
CONFIDENCE_THRESHOLD = 0.8 # min mean-IoU across all 3 pairs → "high confidence"
TOP_N_TRACES       = 12      # number of matched cells to show traces for
REFERENCE_RUN      = 1      # which run's Cn is used as the background image
FS                 = 25.0   # recording frame rate (Hz)

runs = []
for name in RUN_NAMES:
    path = os.path.join(BASE_DIR, name, "caiman_results.hdf5")z
    fit  = cnmf.load_CNMF(path, n_processes=1, dview=None)
    runs.append(fit)
    print(f"{name}:  {len(fit.estimates.idx_components)} accepted  |  "
          f"{len(fit.estimates.idx_components_bad)} rejected")

Cn   = runs[REFERENCE_RUN].estimates.Cn
dims = Cn.shape

# coordinates are not saved in the HDF5 — recompute from the A matrix
from caiman.utils.visualization import get_contours
for run in runs:
    run.estimates.coordinates = get_contours(run.estimates.A, dims)

print(f"\nFrame dims: {dims}  |  background from '{RUN_NAMES[REFERENCE_RUN]}'")

## 2. Component Matching via Spatial Footprint IoU

For each pair of runs we binarise the spatial A-matrix and compute IoU for every
component pair. We then use the Hungarian algorithm for optimal 1-to-1 matching and
keep only pairs above `IOU_THRESHOLD`.  
A **triple match** is a component present in all three runs.

In [ ]:
def build_footprint_masks(fit, dims):
    """Returns (masks: n_acc×H×W bool, ids: 1D int array of component IDs)."""
    idx_acc = np.array(fit.estimates.idx_components)
    A       = fit.estimates.A
    masks   = np.stack(
        [np.asarray(A[:, i].todense()).reshape(dims, order='F') > 0
         for i in idx_acc],
        axis=0,
    )
    return masks, idx_acc


def iou_matrix(masks_a, masks_b):
    """(n_a × n_b) IoU matrix between two sets of H×W boolean footprint masks."""
    a     = masks_a.reshape(len(masks_a), -1).astype(np.float32)  # (na, npix)
    b     = masks_b.reshape(len(masks_b), -1).astype(np.float32)  # (nb, npix)
    inter = a @ b.T                                                 # (na, nb)
    union = a.sum(1, keepdims=True) + b.sum(1, keepdims=True).T - inter
    return np.where(union > 0, inter / union, 0.0)


def hungarian_match(iou_mat, threshold):
    """Optimal 1-to-1 matching; returns (row_idx, col_idx, iou_vals) for pairs >= threshold."""
    row_ind, col_ind = linear_sum_assignment(-iou_mat)
    vals = iou_mat[row_ind, col_ind]
    keep = vals >= threshold
    return row_ind[keep], col_ind[keep], vals[keep]

In [ ]:
all_masks, all_ids = [], []
for run, name in zip(runs, RUN_NAMES):
    masks, ids = build_footprint_masks(run, dims)
    all_masks.append(masks)
    all_ids.append(ids)
    print(f"{name}: {len(ids)} accepted  |  footprint array {masks.shape}")

iou_01 = iou_matrix(all_masks[0], all_masks[1])
iou_02 = iou_matrix(all_masks[0], all_masks[2])
iou_12 = iou_matrix(all_masks[1], all_masks[2])
print(f"\nIoU matrices: 0-1={iou_01.shape}  0-2={iou_02.shape}  1-2={iou_12.shape}")

In [ ]:
r0_01, r1_01, v01 = hungarian_match(iou_01, IOU_THRESHOLD)
r0_02, r2_02, v02 = hungarian_match(iou_02, IOU_THRESHOLD)
r1_12, r2_12, v12 = hungarian_match(iou_12, IOU_THRESHOLD)

map_01 = {r0: (r1, v) for r0, r1, v in zip(r0_01, r1_01, v01)}
map_02 = {r0: (r2, v) for r0, r2, v in zip(r0_02, r2_02, v02)}

rows = []
for r0 in sorted(set(map_01) & set(map_02)):
    r1, v01_val = map_01[r0]
    r2, v02_val = map_02[r0]
    v12_val     = float(iou_12[r1, r2])  # raw IoU between their matched run1/run2 pair
    rows.append({
        'pos0': r0,   'pos1': r1,   'pos2': r2,
        'id0':  int(all_ids[0][r0]),
        'id1':  int(all_ids[1][r1]),
        'id2':  int(all_ids[2][r2]),
        'iou_01':     round(v01_val, 3),
        'iou_02':     round(v02_val, 3),
        'iou_12':     round(v12_val, 3),
        'confidence': round(np.mean([v01_val, v02_val, v12_val]), 3),
    })

df_matches = (
    pd.DataFrame(rows)
    .sort_values('confidence', ascending=False)
    .reset_index(drop=True)
)

print(f"Accepted:       run46={len(all_ids[0])}  run47={len(all_ids[1])}  run57={len(all_ids[2])}")
print(f"Pairwise:       0-1={len(r0_01)}  0-2={len(r0_02)}  1-2={len(r1_12)}")
print(f"Triple matches: {len(df_matches)}")
print(f"High confidence (>= {CONFIDENCE_THRESHOLD}): "
      f"{(df_matches.confidence >= CONFIDENCE_THRESHOLD).sum()}")
df_matches.head(15)

## 3. Spatial Visualization

Three side-by-side FOV maps. Same color = same triple-matched cell. Gray = unmatched.

In [ ]:
N_COLORS   = min(len(df_matches), 20)
disc_cmap  = plt.cm.get_cmap('tab20', N_COLORS)
pos_cols   = ['pos0', 'pos1', 'pos2']
run_labels = [n.replace('caiman_final_', 'run') for n in RUN_NAMES]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for run_i, (ax, run, rlabel, pos_col) in enumerate(
        zip(axes, runs, run_labels, pos_cols)):

    run_Cn = run.estimates.Cn
    vmin, vmax = np.percentile(run_Cn, [1, 99])
    ax.imshow(run_Cn, cmap='gray', vmin=vmin, vmax=vmax)
    matched_pos = set(df_matches[pos_col].values)

    # unmatched → gray
    for pos, comp_id in enumerate(all_ids[run_i]):
        if pos in matched_pos:
            continue
        c = run.estimates.coordinates[comp_id]['coordinates']
        ax.plot(c[:, 0], c[:, 1], color='blue', linewidth=0.6)

    # matched → colored (same color index = same triple-matched cell)
    for g_idx, row in df_matches.iterrows():
        color   = disc_cmap(g_idx) if g_idx < N_COLORS else 'white'
        comp_id = int(all_ids[run_i][int(row[pos_col])])
        c       = run.estimates.coordinates[comp_id]['coordinates']
        ax.plot(c[:, 0], c[:, 1], color=color, linewidth=1.8)

    ax.set_title(f"{rlabel}  —  {len(matched_pos)}/{len(all_ids[run_i])} triple-matched",
                 fontsize=11)
    ax.axis('off')

plt.suptitle(
    "Spatial footprints: same color = matched across all 3 runs  |  gray = unmatched",
    fontsize=12,
)
plt.tight_layout()
plt.show()

## 4. Confidence Analysis

Confidence = mean IoU across all three run-pairs for a triple-matched component.
Higher = more spatially consistent across parameter runs.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(df_matches['confidence'], bins=25, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(CONFIDENCE_THRESHOLD, color='orange', linestyle='--', linewidth=1.5,
           label=f'threshold = {CONFIDENCE_THRESHOLD}')
ax.set_xlabel('Confidence  (mean IoU across 3 run-pairs)')
ax.set_ylabel('Component count')
ax.set_title('Distribution of triple-match confidence')
ax.legend()
plt.tight_layout()
plt.show()

df_high = df_matches[df_matches.confidence >= CONFIDENCE_THRESHOLD].reset_index(drop=True)
print(f"High-confidence components: {len(df_high)}")
df_high[['id0', 'id1', 'id2', 'iou_01', 'iou_02', 'iou_12', 'confidence']]

In [ ]:
# Sanity check: zoom into footprint of the top-confidence match
top_row = df_high.iloc[8]
PAD = 20

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for run_i, (ax, run, id_col, rlabel) in enumerate(
        zip(axes, runs, ['id0', 'id1', 'id2'], run_labels)):
    comp_id  = int(top_row[id_col])
    run_Cn   = run.estimates.Cn
    fp = np.asarray(run.estimates.A[:, comp_id].todense()).reshape(dims, order='F')
    c  = run.estimates.coordinates[comp_id]['coordinates']
    valid = np.isfinite(c[:, 0]) & np.isfinite(c[:, 1])
    r0 = max(0,       int(c[valid, 1].min()) - PAD)
    r1 = min(dims[0], int(c[valid, 1].max()) + PAD)
    c0 = max(0,       int(c[valid, 0].min()) - PAD)
    c1 = min(dims[1], int(c[valid, 0].max()) + PAD)

    ax.imshow(run_Cn[r0:r1, c0:c1], cmap='gray')
    ax.contour(fp[r0:r1, c0:c1] > 0, levels=[0.5], colors='cyan', linewidths=1.5)
    ax.set_title(f"{rlabel}  id={comp_id}\n"
                 f"iou(01={top_row.iou_01:.2f}, 02={top_row.iou_02:.2f}, 12={top_row.iou_12:.2f})",
                 fontsize=9)
    ax.axis('off')

plt.suptitle(f"Top-confidence triple match  (conf={top_row.confidence:.3f})", fontsize=11)
plt.tight_layout()
plt.show()

## 5. Evaluation Scores

Compare r_value, SNR, and CNN score for the same matched cells across the three runs.
Violin plots show population distributions; scatter plots show per-cell consistency.

In [ ]:
def gather_scores(df_subset):
    """Returns dict[run_name][metric] -> 1D array for the matched subset."""
    out = {}
    for run, id_col, rname in zip(runs, ['id0', 'id1', 'id2'], RUN_NAMES):
        ids = df_subset[id_col].values
        cnn = run.estimates.cnn_preds
        out[rname] = {
            'r_value': run.estimates.r_values[ids],
            'SNR':     run.estimates.SNR_comp[ids],
            'CNN':     cnn[ids] if cnn is not None else np.full(len(ids), np.nan),
        }
    return out

score_data = gather_scores(df_high)

In [ ]:
run_colors  = ['#4c72b0', '#dd8452', '#55a868']
metrics     = ['r_value', 'SNR', 'CNN']
metric_lbls = ['Spatial correlation (r)', 'SNR', 'CNN score']

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, metric, label in zip(axes, metrics, metric_lbls):
    data  = [score_data[rname][metric] for rname in RUN_NAMES]
    clean = [d[np.isfinite(d)] for d in data]
    parts = ax.violinplot(clean, positions=range(len(RUN_NAMES)),
                          showmedians=True, showextrema=True)
    for pc, col in zip(parts['bodies'], run_colors):
        pc.set_facecolor(col)
        pc.set_alpha(0.7)
    for key in ('cmedians', 'cmaxes', 'cmins', 'cbars'):
        parts[key].set_color('black')
        parts[key].set_linewidth(1.2)

    # add mean as a diamond marker
    for pos, d in enumerate(clean):
        ax.scatter(pos, np.mean(d), marker='D', color='white',
                   edgecolors='black', s=40, zorder=3, linewidths=0.8)

    ax.set_xticks(range(len(RUN_NAMES)))
    ax.set_xticklabels(run_labels, rotation=15)
    ax.set_ylabel(label)
    ax.set_title(f'{label}\n(n={len(df_high)} cells)')
    ax.grid(axis='y', alpha=0.3)

# legend for median vs mean
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
legend_elements = [
    Line2D([0], [0], color='black', linewidth=1.2, label='median'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='white',
           markeredgecolor='black', markersize=7, label='mean'),
]
axes[-1].legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.suptitle('Evaluation scores for high-confidence matched components', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
from itertools import combinations

pairs     = list(combinations(range(len(RUN_NAMES)), 2))  # (0,1), (0,2), (1,2)
conf      = df_high['confidence'].values

fig, axes = plt.subplots(len(pairs), 3, figsize=(13, 5 * len(pairs)))

for row_i, (i, j) in enumerate(pairs):
    name_i, name_j = RUN_NAMES[i], RUN_NAMES[j]
    id_col_i = ['id0', 'id1', 'id2'][i]
    id_col_j = ['id0', 'id1', 'id2'][j]

    # re-gather scores for this pair's component IDs
    scores_i = {m: runs[i].estimates.__dict__[{'r_value': 'r_values',
                                                'SNR': 'SNR_comp',
                                                'CNN': 'cnn_preds'}[m]]
                for m in metrics}
    scores_j = {m: runs[j].estimates.__dict__[{'r_value': 'r_values',
                                                'SNR': 'SNR_comp',
                                                'CNN': 'cnn_preds'}[m]]
                for m in metrics}

    ids_i = df_high[id_col_i].values.astype(int)
    ids_j = df_high[id_col_j].values.astype(int)

    for col_i, (metric, label) in enumerate(zip(metrics, metric_lbls)):
        ax = axes[row_i, col_i]

        si = scores_i[metric]
        sj = scores_j[metric]

        # handle cnn_preds being None
        arr_i = si[ids_i] if si is not None else np.full(len(ids_i), np.nan)
        arr_j = sj[ids_j] if sj is not None else np.full(len(ids_j), np.nan)

        valid = np.isfinite(arr_i) & np.isfinite(arr_j)
        sc = ax.scatter(arr_i[valid], arr_j[valid],
                        c=conf[valid], cmap='plasma', s=40, alpha=0.8,
                        vmin=CONFIDENCE_THRESHOLD, vmax=1.0)
        lo  = min(arr_i[valid].min(), arr_j[valid].min())
        hi  = max(arr_i[valid].max(), arr_j[valid].max())
        pad = (hi - lo) * 0.05
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad],
                'k--', linewidth=0.8, alpha=0.5)
        ax.set_xlim(lo - pad, hi + pad)
        ax.set_ylim(lo - pad, hi + pad)
        ax.set_xlabel(f"{name_i.split('_')[-1]}  {label}")
        ax.set_ylabel(f"{name_j.split('_')[-1]}  {label}")
        ax.set_title(label)
        ax.grid(alpha=0.3)
        plt.colorbar(sc, ax=ax, label='confidence')

plt.suptitle(
    'Per-component scores — all pairwise run comparisons\n'
    'each point = one matched cell  (color = confidence)',
    fontsize=12,
)
plt.tight_layout()
plt.show()

## 6. Trace & Noise Comparison

For the top-N most confident matched cells, overlay ΔF/F traces from all three runs
(normalised per-trace so amplitude differences don't obscure shape).  
Noise is estimated as the MAD of first differences of the ΔF/F (or from YrA residuals
if available), which captures the unexplained high-frequency variance.

## 5b. Evaluation Scores — All Components (Accepted & Rejected)

Same metrics as above but for the **full population** of each run, split by accepted/rejected.
Useful for understanding how the parameter set shifts the acceptance boundary, independently of cross-run matching.

In [ ]:
acc_color = '#2ca02c'   # green  — accepted
rej_color = '#d62728'   # red    — rejected

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

attr_map = {'r_value': 'r_values', 'SNR': 'SNR_comp', 'CNN': 'cnn_preds'}

# x positions: for each run pair [acc, rej], spaced with a gap between runs
positions, xlabels, xcolors = [], [], []
for run_i in range(len(RUN_NAMES)):
    base = run_i * 3
    positions.append((base, base + 1))          # (acc_pos, rej_pos)
    short = RUN_NAMES[run_i].replace('caiman_final_', 'run')
    xlabels  += [f'{short}\nacc', f'{short}\nrej']
    xcolors  += [acc_color, rej_color]

for ax, metric, label in zip(axes, metrics, metric_lbls):
    attr = attr_map[metric]
    all_data, all_pos = [], []

    for run_i, run in enumerate(runs):
        arr = getattr(run.estimates, attr)
        if arr is None:
            acc_data = np.full(len(run.estimates.idx_components), np.nan)
            rej_data = np.full(len(run.estimates.idx_components_bad), np.nan)
        else:
            acc_data = arr[np.array(run.estimates.idx_components)]
            rej_data = arr[np.array(run.estimates.idx_components_bad)]

        acc_clean = acc_data[np.isfinite(acc_data)]
        rej_clean = rej_data[np.isfinite(rej_data)]
        acc_pos, rej_pos = positions[run_i]

        for data, pos, color in [(acc_clean, acc_pos, acc_color),
                                  (rej_clean, rej_pos, rej_color)]:
            if len(data) < 2:
                continue
            parts = ax.violinplot([data], positions=[pos],
                                  showmedians=True, showextrema=True)
            parts['bodies'][0].set_facecolor(color)
            parts['bodies'][0].set_alpha(0.65)
            for key in ('cmedians', 'cmaxes', 'cmins', 'cbars'):
                parts[key].set_color('black')
                parts[key].set_linewidth(1.0)
            ax.scatter(pos, np.mean(data), marker='D', color='white',
                       edgecolors='black', s=30, zorder=3, linewidths=0.7)

    # vertical separators between runs
    for run_i in range(1, len(RUN_NAMES)):
        ax.axvline(run_i * 3 - 0.5, color='gray', linewidth=0.8,
                   linestyle='--', alpha=0.5)

    flat_pos = [p for pair in positions for p in pair]
    ax.set_xticks(flat_pos)
    ax.set_xticklabels(xlabels, fontsize=8)
    for tick, col in zip(ax.get_xticklabels(), xcolors):
        tick.set_color(col)
    ax.set_ylabel(label)
    n_acc = [len(run.estimates.idx_components) for run in runs]
    n_rej = [len(run.estimates.idx_components_bad) for run in runs]
    ax.set_title(f'{label}\nacc: {n_acc}  |  rej: {n_rej}', fontsize=9)
    ax.grid(axis='y', alpha=0.3)

from matplotlib.patches import Patch
from matplotlib.lines import Line2D
legend_elements = [
    Patch(facecolor=acc_color, alpha=0.65, label='accepted'),
    Patch(facecolor=rej_color, alpha=0.65, label='rejected'),
    Line2D([0], [0], color='black', linewidth=1.2, label='median'),
    Line2D([0], [0], marker='D', color='w', markerfacecolor='white',
           markeredgecolor='black', markersize=7, label='mean'),
]
axes[-1].legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.suptitle('Evaluation scores — all components per run  (accepted vs rejected)',
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# from mpl_toolkits.mplot3d import Axes3D
# from matplotlib.animation import FuncAnimation
# from IPython.display import HTML

# fig = plt.figure(figsize=(18, 6))
# ax_list = []

# for run_i, (run, rlabel) in enumerate(zip(runs, run_labels)):
#     ax = fig.add_subplot(1, 3, run_i + 1, projection='3d')
#     ax_list.append(ax)

#     r_vals = run.estimates.r_values
#     snr    = run.estimates.SNR_comp
#     cnn    = run.estimates.cnn_preds

#     for idx_arr, color, status in [
#         (run.estimates.idx_components,     acc_color, 'accepted'),
#         (run.estimates.idx_components_bad, rej_color, 'rejected'),
#     ]:
#         ids   = np.array(idx_arr)
#         x     = r_vals[ids]
#         y     = snr[ids]
#         z     = cnn[ids] if cnn is not None else np.zeros(len(ids))
#         valid = np.isfinite(x) & np.isfinite(y) & np.isfinite(z)
#         ax.scatter(x[valid], y[valid], z[valid],
#                    c=color, alpha=0.6, s=20, label=status, depthshade=True)

#     ax.set_xlabel('r value',  fontsize=8, labelpad=4)
#     ax.set_ylabel('SNR',      fontsize=8, labelpad=4)
#     ax.set_zlabel('CNN',      fontsize=8, labelpad=4)
#     ax.set_title(rlabel, fontsize=10)
#     ax.tick_params(labelsize=7)

# ax_list[0].legend(fontsize=8, loc='upper left')
# plt.suptitle('Component scores in 3D  (r_value × SNR × CNN)', fontsize=12)
# plt.tight_layout()

# def _rotate(frame):
#     for ax in ax_list:
#         ax.view_init(elev=20, azim=frame)

# anim = FuncAnimation(fig, _rotate, frames=np.arange(0, 360, 2),
#                      interval=50, blit=False)
# plt.close()
# HTML(anim.to_jshtml())

In [ ]:
def load_trace_and_noise(run, comp_id):
    """Returns (F_dff trace 1D, noise std).
    Noise: std of YrA residuals if available; else MAD of first differences."""
    trace = (run.estimates.F_dff[comp_id, :]
             if run.estimates.F_dff is not None
             else run.estimates.C[comp_id, :])
    if run.estimates.YrA is not None:
        noise = float(np.std(run.estimates.YrA[comp_id, :]))
    else:
        # robust noise from first differences; sqrt(2) corrects for differencing
        noise = float(np.median(np.abs(np.diff(trace))) / 0.6745 / np.sqrt(2))
    return trace, noise

In [ ]:
top_df = df_high.head(TOP_N_TRACES)

fig, axes = plt.subplots(len(top_df), 1, figsize=(15, 2.8 * len(top_df)))
if len(top_df) == 1:
    axes = [axes]

for ax_i, (_, row) in enumerate(top_df.iterrows()):
    ax         = axes[ax_i]
    noise_vals = []

    for run_i, (run, id_col, color, rlabel) in enumerate(
            zip(runs, ['id0', 'id1', 'id2'], run_colors, run_labels)):
        comp_id = int(row[id_col])
        trace, noise = load_trace_and_noise(run, comp_id)
        noise_vals.append(noise)

        # normalise to [0, 1] then stack with offset for readability
        lo, hi  = trace.min(), trace.max()
        tr_norm = (trace - lo) / max(hi - lo, 1e-9)
        time    = np.arange(len(trace)) / FS
        ax.plot(time, tr_norm + run_i * 1.15, color=color, linewidth=0.6,
                label=f"{rlabel}  noise={noise:.3f}")

    ax.set_yticks([0.5 + i * 1.15 for i in range(len(runs))])
    ax.set_yticklabels(run_labels, fontsize=8)
    ax.set_xlabel('Time (s)' if ax_i == len(top_df) - 1 else '')
    ax.set_title(
        f"Match #{ax_i + 1}  conf={row.confidence:.3f}  "
        f"ids: {int(row.id0)}/{int(row.id1)}/{int(row.id2)}  "
        f"noise: {'/'.join(f'{n:.3f}' for n in noise_vals)}",
        fontsize=9,
    )
    ax.grid(axis='x', alpha=0.25)

plt.suptitle(
    f'ΔF/F traces — top {len(top_df)} high-confidence matched cells\n'
    f'(each trace normalised to [0, 1] independently; stacked for visibility)',
    fontsize=11,
)
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import skew as compute_skew
from itertools import combinations as _combinations

# ── collect per-cell metrics for all high-confidence matched cells ────────────
noise_per_run = {rname: [] for rname in RUN_NAMES}
skew_per_run  = {rname: [] for rname in RUN_NAMES}

for _, row in df_high.iterrows():
    for run, id_col, rname in zip(runs, ['id0', 'id1', 'id2'], RUN_NAMES):
        trace, noise = load_trace_and_noise(run, int(row[id_col]))
        noise_per_run[rname].append(noise)
        skew_per_run[rname].append(float(compute_skew(trace)))

# cross-run trace correlation for each matched pair
pair_labels, pair_corrs = [], []
for i, j in _combinations(range(len(RUN_NAMES)), 2):
    id_col_i = ['id0', 'id1', 'id2'][i]
    id_col_j = ['id0', 'id1', 'id2'][j]
    corrs = []
    for _, row in df_high.iterrows():
        ti, _ = load_trace_and_noise(runs[i], int(row[id_col_i]))
        tj, _ = load_trace_and_noise(runs[j], int(row[id_col_j]))
        corrs.append(float(np.corrcoef(ti, tj)[0, 1]))
    pair_labels.append(f"{run_labels[i]}\nvs {run_labels[j]}")
    pair_corrs.append(corrs)

# ── figure ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

def _violin(ax, data_dict, keys, colors, ylabel, title, add_mean=True):
    data  = [data_dict[k] for k in keys]
    parts = ax.violinplot(data, positions=range(len(keys)),
                          showmedians=True, showextrema=True)
    for pc, col in zip(parts['bodies'], colors):
        pc.set_facecolor(col)
        pc.set_alpha(0.7)
    for key in ('cmedians', 'cmaxes', 'cmins', 'cbars'):
        parts[key].set_color('black')
        parts[key].set_linewidth(1.2)
    if add_mean:
        for pos, d in enumerate(data):
            ax.scatter(pos, np.mean(d), marker='D', color='white',
                       edgecolors='black', s=35, zorder=3, linewidths=0.8)
    ax.set_xticks(range(len(keys)))
    ax.set_xticklabels(keys, fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)

_violin(axes[0], noise_per_run, RUN_NAMES,
        run_colors,
        'Noise std  (MAD of ΔF/F diff)',
        f'Residual noise\n(n={len(df_high)} matched cells)')
axes[0].set_xticklabels(run_labels, fontsize=9)

_violin(axes[1], skew_per_run, RUN_NAMES,
        run_colors,
        'ΔF/F skewness',
        'Trace skewness\n(higher = cleaner transients)')
axes[1].set_xticklabels(run_labels, fontsize=9)
axes[1].axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.6)

# cross-run trace correlation — separate colors per pair
pair_colors = ['#9467bd', '#8c564b', '#e377c2']
parts = axes[2].violinplot(pair_corrs, positions=range(len(pair_labels)),
                           showmedians=True, showextrema=True)
for pc, col in zip(parts['bodies'], pair_colors):
    pc.set_facecolor(col)
    pc.set_alpha(0.7)
for key in ('cmedians', 'cmaxes', 'cmins', 'cbars'):
    parts[key].set_color('black')
    parts[key].set_linewidth(1.2)
for pos, d in enumerate(pair_corrs):
    axes[2].scatter(pos, np.mean(d), marker='D', color='white',
                    edgecolors='black', s=35, zorder=3, linewidths=0.8)
axes[2].set_xticks(range(len(pair_labels)))
axes[2].set_xticklabels(pair_labels, fontsize=9)
axes[2].set_ylabel('Pearson r  (ΔF/F traces)')
axes[2].set_title('Cross-run trace correlation\n(same cell, different run)')
axes[2].set_ylim(-0.1, 1.05)
axes[2].axhline(0, color='gray', linewidth=0.8, linestyle='--', alpha=0.6)
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Signal quality for high-confidence matched cells', fontsize=12)
plt.tight_layout()
plt.show()